# Phase 4B — Minimal ATE Train-Val Smoke Run

**Important caveat**  
This notebook intentionally ignores the previously identified Phase 2 ATE supervision mismatch issue (the small subset of substring-based matched aspects that are not fully preserved in BIO export).

The goal of this notebook is **not** to produce final ATE results.  
Its purpose is only to verify that the current ATE branch is:

- loadable,
- tokenizable,
- batchable,
- trainable,
- and validation-ready.

In other words, this notebook is a **trainability smoke test**, not a final experiment notebook.

GAO YU: Currently is 1e-6 & SGD. At this moment, I just want to test if it is workable, but for real training should not be 1e-6, also, it should be adam/W

LR = 1e-6

WEIGHT_DECAY = 0.0

FREEZE_BACKBONE_FOR_SMOKE = True

In [1]:
import copy
import json
from pathlib import Path
from pprint import pprint

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForTokenClassification

d:\anaconda3\envs\tpwng\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = Path("outputs_phase2")
META_DIR = Path("outputs_phase3")
OUT_DIR = Path("outputs_phase4_ate_smoke")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ATE_TRAIN_PATH = DATA_DIR / "ate_train.jsonl"
ATE_VAL_PATH   = DATA_DIR / "ate_val.jsonl"
ATE_TEST_PATH  = DATA_DIR / "ate_test.jsonl"

SPLIT_STATS_PATH = DATA_DIR / "split_stats_report.json"
ATE_SMOKE_PHASE3_PATH = META_DIR / "ate_smoke_report.json"

SMOKE_REPORT_PATH = OUT_DIR / "ate_min_train_val_smoke_report.json"

BACKBONE_NAME = "microsoft/deberta-v3-base"
# Optional lighter fallback:
# BACKBONE_NAME = "microsoft/deberta-v3-small"

MAX_LEN = 128
BATCH_SIZE = 4
LR = 1e-6
WEIGHT_DECAY = 0.0
FREEZE_BACKBONE_FOR_SMOKE = True

SMOKE_TRAIN_SIZE = 256
SMOKE_VAL_SIZE = 64
MAX_TRAIN_STEPS = 20
MAX_VAL_BATCHES = 5

ATE_LABEL2ID = {
    "O": 0,
    "B-ASP": 1,
    "I-ASP": 2
}
ATE_ID2LABEL = {v: k for k, v in ATE_LABEL2ID.items()}

IGNORE_INDEX = -100
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

In [3]:
with open(SPLIT_STATS_PATH, "r", encoding="utf-8") as f:
    split_stats = json.load(f)

with open(ATE_SMOKE_PHASE3_PATH, "r", encoding="utf-8") as f:
    ate_phase3_report = json.load(f)

print("Phase 2 ATE validation:")
pprint(split_stats["ate_validation"])

print("\nPhase 3 ATE smoke report:")
pprint(ate_phase3_report)

Phase 2 ATE validation:
{'test_rows': 1079,
 'test_token_label_len_ok': True,
 'train_rows': 8576,
 'train_token_label_len_ok': True,
 'val_rows': 1071,
 'val_token_label_len_ok': True}

Phase 3 ATE smoke report:
{'backbone': 'microsoft/deberta-v3-base',
 'batch_shape_attention_mask': [4, 14],
 'batch_shape_input_ids': [4, 14],
 'batch_shape_labels': [4, 14],
 'first_sample_token_label_len_match': True,
 'sample_encoded_len': 13,
 'sample_word_len': 10,
 'test_examples': 1079,
 'train_examples': 8576,
 'val_examples': 1071}


In [4]:
def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return records

In [5]:
ate_train = load_jsonl(ATE_TRAIN_PATH)
ate_val   = load_jsonl(ATE_VAL_PATH)
ate_test  = load_jsonl(ATE_TEST_PATH)

print("ATE train/val/test sizes:", len(ate_train), len(ate_val), len(ate_test))

ATE train/val/test sizes: 8576 1071 1079


In [6]:
required_fields = [
    "doc_id", "raw_id", "split", "text",
    "tokens", "token_offsets", "bio_labels", "aspects"
]

sample = ate_train[0]
missing = [f for f in required_fields if f not in sample]

print("Missing fields:", missing)
print("Field validation passed:", len(missing) == 0)
pprint(sample)

Missing fields: []
Field validation passed: True
{'aspects': [{'char_from': 0,
              'char_to': 8,
              'match_status': 'matched',
              'match_type': 'whole_word_unique',
              'sentiment': 'neutral',
              'term': 'SpiceJet'}],
 'bio_labels': ['B-ASP', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'],
 'doc_id': 'sentfin_000001',
 'raw_id': 1,
 'split': 'train',
 'text': 'SpiceJet to issue 6.4 crore warrants to promoters',
 'token_offsets': [[0, 8],
                   [9, 11],
                   [12, 17],
                   [18, 19],
                   [19, 20],
                   [20, 21],
                   [22, 27],
                   [28, 36],
                   [37, 39],
                   [40, 49]],
 'tokens': ['SpiceJet',
            'to',
            'issue',
            '6',
            '.',
            '4',
            'crore',
            'warrants',
            'to',
            'promoters']}


In [7]:
train_smoke = ate_train[:SMOKE_TRAIN_SIZE]
val_smoke = ate_val[:SMOKE_VAL_SIZE]

print("Smoke train size:", len(train_smoke))
print("Smoke val size:", len(val_smoke))

Smoke train size: 256
Smoke val size: 64


In [8]:
tokenizer = AutoTokenizer.from_pretrained(BACKBONE_NAME)

print("Loaded tokenizer:", BACKBONE_NAME)
print("Pad token:", tokenizer.pad_token)
print("CLS token:", tokenizer.cls_token)
print("SEP token:", tokenizer.sep_token)

Loaded tokenizer: microsoft/deberta-v3-base
Pad token: [PAD]
CLS token: [CLS]
SEP token: [SEP]


In [9]:
ate_sample = train_smoke[0]

print("Text:", ate_sample["text"])
print("Tokens:", ate_sample["tokens"])
print("BIO labels:", ate_sample["bio_labels"])
print("Lengths:", len(ate_sample["tokens"]), len(ate_sample["bio_labels"]))

Text: SpiceJet to issue 6.4 crore warrants to promoters
Tokens: ['SpiceJet', 'to', 'issue', '6', '.', '4', 'crore', 'warrants', 'to', 'promoters']
BIO labels: ['B-ASP', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
Lengths: 10 10


In [10]:
def encode_ate_example(example, tokenizer, max_length=128):
    tokens = example["tokens"]
    word_labels = example["bio_labels"]

    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=max_length,
        padding=False
    )

    word_ids = encoding.word_ids()
    aligned_labels = []
    prev_word_id = None

    for word_id in word_ids:
        if word_id is None:
            aligned_labels.append(IGNORE_INDEX)
        elif word_id != prev_word_id:
            aligned_labels.append(ATE_LABEL2ID[word_labels[word_id]])
        else:
            aligned_labels.append(IGNORE_INDEX)
        prev_word_id = word_id

    encoding["labels"] = aligned_labels
    return encoding

In [11]:
ate_encoded = encode_ate_example(ate_sample, tokenizer, max_length=MAX_LEN)

print("input_ids length:", len(ate_encoded["input_ids"]))
print("attention_mask length:", len(ate_encoded["attention_mask"]))
print("labels length:", len(ate_encoded["labels"]))
print("word_ids:", ate_encoded.word_ids()[:30])
print("labels:", ate_encoded["labels"][:30])

input_ids length: 13
attention_mask length: 13
labels length: 13
word_ids: [None, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, None]
labels: [-100, 1, -100, 0, 0, 0, 0, 0, 0, 0, 0, 0, -100]


In [12]:
def pretty_print_ate_alignment(example, encoding, tokenizer, max_rows=60):
    toks = tokenizer.convert_ids_to_tokens(encoding["input_ids"])
    word_ids = encoding.word_ids()
    labels = encoding["labels"]

    rows = []
    for i, (tok, wid, lab) in enumerate(zip(toks, word_ids, labels)):
        rows.append({
            "idx": i,
            "subword": tok,
            "word_id": wid,
            "label_id": lab,
            "label_name": None if lab == IGNORE_INDEX else ATE_ID2LABEL[lab]
        })

    return pd.DataFrame(rows).head(max_rows)

In [13]:
pretty_print_ate_alignment(ate_sample, ate_encoded, tokenizer)

,idx,subword,word_id,label_id,label_name
0,0,[CLS],NaN,-100,NaN
1,1,▁Spice,0.0,1,B-ASP
2,2,Jet,0.0,-100,NaN
3,3,▁to,1.0,0,O
4,4,▁issue,2.0,0,O
5,5,▁6,3.0,0,O
6,6,▁.,4.0,0,O
7,7,▁4,5.0,0,O
8,8,▁crore,6.0,0,O
9,9,▁warrants,7.0,0,O


In [14]:
class ATESmokeDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        return self.records[idx]

In [15]:
train_dataset = ATESmokeDataset(train_smoke)
val_dataset = ATESmokeDataset(val_smoke)

print("Train dataset size:", len(train_dataset))
print("Val dataset size:", len(val_dataset))

Train dataset size: 256
Val dataset size: 64


In [16]:
def ate_collate_fn(batch):
    encodings = [encode_ate_example(ex, tokenizer, max_length=MAX_LEN) for ex in batch]

    labels_list = [enc.pop("labels") for enc in encodings]

    batch_enc = tokenizer.pad(
        encodings,
        padding=True,
        return_tensors="pt"
    )

    max_len = batch_enc["input_ids"].shape[1]
    padded_labels = []

    for labels in labels_list:
        padded = labels + [IGNORE_INDEX] * (max_len - len(labels))
        padded_labels.append(padded)

    batch_enc["labels"] = torch.tensor(padded_labels, dtype=torch.long)
    return batch_enc

In [17]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=ate_collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=ate_collate_fn
)

In [18]:
train_batch = next(iter(train_loader))

print("input_ids shape:", train_batch["input_ids"].shape)
print("attention_mask shape:", train_batch["attention_mask"].shape)
print("labels shape:", train_batch["labels"].shape)

input_ids shape: torch.Size([4, 13])
attention_mask shape: torch.Size([4, 13])
labels shape: torch.Size([4, 13])


In [19]:
model = AutoModelForTokenClassification.from_pretrained(
    BACKBONE_NAME,
    num_labels=3,
    id2label=ATE_ID2LABEL,
    label2id=ATE_LABEL2ID
)

if FREEZE_BACKBONE_FOR_SMOKE:
    for param in model.parameters():
        param.requires_grad = False

    for name, param in model.named_parameters():
        if name.startswith("classifier"):
            param.requires_grad = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Device:", device)
print("Model loaded:", BACKBONE_NAME)
print("Freeze backbone for smoke:", FREEZE_BACKBONE_FOR_SMOKE)
print("Trainable params:", trainable_params)

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 15791.45it/s]
DebertaV2ForTokenClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 

Notes:
- UNEX

Device: cuda
Model loaded: microsoft/deberta-v3-base
Freeze backbone for smoke: True
Trainable params: 2307


In [20]:
optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR
)

In [21]:
batch_for_forward = {k: v.to(device) for k, v in train_batch.items()}

model.train()
outputs = model(**batch_for_forward)

print("Loss:", float(outputs.loss))
print("Logits shape:", outputs.logits.shape)

Loss: 1.1044921875
Logits shape: torch.Size([4, 13, 3])


In [22]:
optimizer.zero_grad()
outputs.loss.backward()
optimizer.zero_grad()

print("One backward pass completed successfully.")

One backward pass completed successfully.


In [23]:
# Start the tiny loop from a clean model state after the one-batch
# forward/backward smoke check above.
model = AutoModelForTokenClassification.from_pretrained(
    BACKBONE_NAME,
    num_labels=3,
    id2label=ATE_ID2LABEL,
    label2id=ATE_LABEL2ID
)

if FREEZE_BACKBONE_FOR_SMOKE:
    for param in model.parameters():
        param.requires_grad = False

    for name, param in model.named_parameters():
        if name.startswith("classifier"):
            param.requires_grad = True

model.to(device)

optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR
)

last_finite_state = copy.deepcopy(model.state_dict())

train_losses = []
train_steps = 0

model.train()

for batch in train_loader:
    batch = {k: v.to(device) for k, v in batch.items()}

    optimizer.zero_grad()
    step_start_state = copy.deepcopy(model.state_dict())
    outputs = model(**batch)
    loss = outputs.loss

    if not torch.isfinite(loss) or not torch.isfinite(outputs.logits).all():
        print(f"Train step {train_steps + 1:02d} | non-finite loss encountered, stopping early.")
        break

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    params_finite = all(torch.isfinite(p).all() for p in model.parameters() if p.requires_grad)
    if not params_finite:
        model.load_state_dict(step_start_state)
        print(f"Train step {train_steps + 1:02d} | parameters became non-finite after update, reverting and stopping early.")
        break

    train_losses.append(float(loss.detach().cpu()))
    last_finite_state = copy.deepcopy(model.state_dict())
    train_steps += 1

    print(f"Train step {train_steps:02d} | loss = {train_losses[-1]:.4f}")

    if train_steps >= MAX_TRAIN_STEPS:
        break

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 26101.52it/s]
DebertaV2ForTokenClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 

Notes:
- UNEX

Train step 01 | loss = 1.6045
Train step 02 | loss = 1.7754
Train step 03 | loss = 1.6338
Train step 04 | loss = 1.6768
Train step 05 | loss = 1.7783
Train step 06 | loss = 1.6895
Train step 07 | loss = 1.7705
Train step 08 | loss = 1.8018
Train step 09 | loss = 1.6895
Train step 10 | loss = 1.8320
Train step 11 | loss = 1.7891
Train step 12 | loss = 1.7090
Train step 13 | loss = 1.7148
Train step 14 | loss = 1.6514
Train step 15 | loss = 1.7090
Train step 16 | loss = 1.5889
Train step 17 | loss = 1.6416
Train step 18 | loss = 1.8018
Train step 19 | loss = 1.5889
Train step 20 | loss = 1.7559


In [24]:
def masked_token_accuracy(logits, labels, ignore_index=-100):
    preds = torch.argmax(logits, dim=-1)
    mask = labels != ignore_index

    if mask.sum().item() == 0:
        return 0.0, 0, 0

    correct = ((preds == labels) & mask).sum().item()
    total = mask.sum().item()
    return correct / total, correct, total

In [25]:
# Restore the last known finite weights before validation so that
# evaluation remains runnable even if a later train step diverged.
model.load_state_dict(last_finite_state)

val_losses = []
val_accs = []
val_correct = 0
val_total = 0
val_batches_seen = 0

model.eval()

with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss
        logits = outputs.logits
        labels = batch["labels"]

        if not torch.isfinite(loss) or not torch.isfinite(logits).all():
            print(f"Val batch {val_batches_seen + 1:02d} | non-finite loss encountered, stopping early.")
            break

        acc, correct, total = masked_token_accuracy(logits, labels, ignore_index=IGNORE_INDEX)

        val_losses.append(float(loss.detach().cpu()))
        val_accs.append(acc)
        val_correct += correct
        val_total += total
        val_batches_seen += 1

        print(f"Val batch {val_batches_seen:02d} | loss = {val_losses[-1]:.4f} | masked_acc = {acc:.4f}")

        if val_batches_seen >= MAX_VAL_BATCHES:
            break

Val batch 01 | loss = 1.6445 | masked_acc = 0.0968
Val batch 02 | loss = 1.6582 | masked_acc = 0.0909
Val batch 03 | loss = 1.7402 | masked_acc = 0.0270
Val batch 04 | loss = 1.7822 | masked_acc = 0.0833
Val batch 05 | loss = 1.7832 | masked_acc = 0.0000


In [26]:
train_loss_mean = float(np.mean(train_losses)) if len(train_losses) > 0 else None
val_loss_mean = float(np.mean(val_losses)) if len(val_losses) > 0 else None
val_token_acc = float(val_correct / val_total) if val_total > 0 else None

print("Mean train loss:", train_loss_mean)
print("Mean val loss:", val_loss_mean)
print("Val masked token accuracy:", val_token_acc)

Mean train loss: 1.710107421875
Mean val loss: 1.7216796875
Val masked token accuracy: 0.06097560975609756


In [27]:
smoke_report = {
    "backbone": BACKBONE_NAME,
    "device": str(device),
    "train_records_total": len(ate_train),
    "val_records_total": len(ate_val),
    "test_records_total": len(ate_test),
    "smoke_train_size": len(train_smoke),
    "smoke_val_size": len(val_smoke),
    "batch_size": BATCH_SIZE,
    "max_len": MAX_LEN,
    "freeze_backbone_for_smoke": FREEZE_BACKBONE_FOR_SMOKE,
    "max_train_steps": MAX_TRAIN_STEPS,
    "max_val_batches": MAX_VAL_BATCHES,
    "one_batch_input_shape": list(train_batch["input_ids"].shape),
    "one_batch_label_shape": list(train_batch["labels"].shape),
    "forward_pass_ok": True,
    "backward_pass_ok": True,
    "train_steps_completed": train_steps,
    "val_batches_completed": val_batches_seen,
    "mean_train_loss": train_loss_mean,
    "mean_val_loss": val_loss_mean,
    "val_masked_token_accuracy": val_token_acc
}

with open(SMOKE_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(smoke_report, f, ensure_ascii=False, indent=2)

pprint(smoke_report)
print("Saved smoke report to:", SMOKE_REPORT_PATH)

{'backbone': 'microsoft/deberta-v3-base',
 'backward_pass_ok': True,
 'batch_size': 4,
 'device': 'cuda',
 'forward_pass_ok': True,
 'freeze_backbone_for_smoke': True,
 'max_len': 128,
 'max_train_steps': 20,
 'max_val_batches': 5,
 'mean_train_loss': 1.710107421875,
 'mean_val_loss': 1.7216796875,
 'one_batch_input_shape': [4, 13],
 'one_batch_label_shape': [4, 13],
 'smoke_train_size': 256,
 'smoke_val_size': 64,
 'test_records_total': 1079,
 'train_records_total': 8576,
 'train_steps_completed': 20,
 'val_batches_completed': 5,
 'val_masked_token_accuracy': 0.06097560975609756,
 'val_records_total': 1071}
Saved smoke report to: outputs_phase4_ate_smoke\ate_min_train_val_smoke_report.json


In [28]:
checklist = [
    ("ATE train file loaded", len(ate_train) > 0),
    ("ATE val file loaded", len(ate_val) > 0),
    ("Tokenizer loaded", tokenizer is not None),
    ("One train batch created", train_batch["input_ids"].shape[0] > 0),
    ("Forward pass completed", True),
    ("Backward pass completed", True),
    ("At least one train step completed", train_steps > 0),
    ("At least one val batch completed", val_batches_seen > 0),
    ("Smoke report saved", SMOKE_REPORT_PATH.exists()),
]

check_df = pd.DataFrame(checklist, columns=["check_item", "passed"])
display(check_df)

print("ATE MINIMAL SMOKE RUN COMPLETE:", check_df["passed"].all())

,check_item,passed
0,ATE train file loaded,True
1,ATE val file loaded,True
2,Tokenizer loaded,True
3,One train batch created,True
4,Forward pass completed,True
5,Backward pass completed,True
6,At least one train step completed,True
7,At least one val batch completed,True
8,Smoke report saved,True


ATE MINIMAL SMOKE RUN COMPLETE: True


## Phase 4B conclusion

If all checks pass, then the current ATE branch has been verified as trainable at the engineering level.

This means:
- the Phase 2 ATE exports can be consumed by the model,
- the Phase 3 tokenizer/alignment logic is workable,
- and a minimal training-validation loop can be executed successfully.

This notebook does **not** resolve the known Phase 2 ATE supervision mismatch issue.
It only proves that the current ATE branch is operational enough for further controlled development.